# ADA — Real-Data Pretraining (FineWeb-Edu)

**This notebook starts an actual causal-language-model pretraining run from randomly initialized ADA weights on streamed public text. It is not a smoke test.**

### What happens
1. Connect a Colab GPU and install ADA's pretraining dependencies.
2. Stream real English educational documents from `HuggingFaceFW/fineweb-edu`, config `sample-10BT`.
3. Inspect real source documents and metadata before training.
4. Train a byte-level BPE tokenizer on a bounded training-only sample.
5. Train ADA's 4-layer causal LM with Mamba-2 reference SSD / causal attention alternation, sparse MoE and Energy Memory.
6. Evaluate on a deterministic held-out document split, save resumable checkpoints, plot loss, and sample generation.

### First-run budget
- 500 optimizer steps × 4 sequences × 256 next-token targets = **512,000 training tokens**.
- 8,192-token BPE vocabulary; 128 hidden width; 4 decoder layers.
- FP32 on the available Colab GPU for numerical reliability in this first real-data run.
- Streaming ingestion: no full 10BT corpus download and no dataset files committed to Git.

This is a **bounded pretraining pilot**, not a frontier-scale training run. Its purpose is to produce a real-data-trained checkpoint and validation curve. Scale only after inspecting throughput, loss, data quality, and stability.

### Data provenance and usage
- Dataset: [HuggingFaceFW/fineweb-edu](https://huggingface.co/datasets/HuggingFaceFW/fineweb-edu), config `sample-10BT`, split `train`, pinned revision `87f09149ef4734204d70ed1d046ddc9ca3f2b8f9`.
- Declared license: ODC-By 1.0. The dataset card also says Common Crawl terms apply. Preserve attribution and review the source terms before redistribution.
- FineWeb-Edu is English educational web text; it is not a balanced multilingual, code, cybersecurity, or domain-specialist corpus. Its dataset card warns that residual toxic content and web-data biases may remain.
- A stable SHA-256 bucket split reserves 1% of records for validation. Training tokenizer and training batches use only the remaining 99%.

Source: [FineWeb-Edu dataset card](https://huggingface.co/datasets/HuggingFaceFW/fineweb-edu) · [FineWeb-Edu paper](https://arxiv.org/abs/2406.17557)

In [ ]:
import os, subprocess, sys, json, time
from pathlib import Path
import torch

if not torch.cuda.is_available():
    raise RuntimeError('GPU required: Colab → Runtime → Change runtime type → T4 GPU (or better), then rerun.')
print('GPU:', torch.cuda.get_device_name(0))
print('PyTorch:', torch.__version__, '| CUDA:', torch.version.cuda)

REPO_DIR = Path('/content/ADA')
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', 'main', 'https://github.com/vimalspaceton618-afk/ADA.git', str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[pretraining]'], check=True)
print('ADA + streaming pretraining dependencies installed from main.')

In [ ]:
# Persist tokenizer, metrics, and checkpoints across Colab sessions.
from google.colab import drive
drive.mount('/content/drive')
RUN_DIR = Path('/content/drive/MyDrive/ADA_Pretraining/fineweb_edu_pilot')
RUN_DIR.mkdir(parents=True, exist_ok=True)
print('Run artifacts:', RUN_DIR)

## 1. Inspect the real training source

The next cell reads a few records from the streaming dataset and displays source text, score, language, document ID, and source URL. It does not download the full dataset.

In [ ]:
from datasets import load_dataset
from IPython.display import display, Markdown

DATASET_ID = 'HuggingFaceFW/fineweb-edu'
DATASET_CONFIG = 'sample-10BT'
raw_stream = load_dataset(DATASET_ID, name=DATASET_CONFIG, split='train', revision='87f09149ef4734204d70ed1d046ddc9ca3f2b8f9', streaming=True)
for i, row in enumerate(raw_stream.take(3), start=1):
    print(f"DOCUMENT {i} | language={row.get('language')} | int_score={row.get('int_score')} | token_count={row.get('token_count')}")
    print('ID:', row.get('id'))
    print('URL:', row.get('url'))
    print((row.get('text') or '')[:1400], '\\n')
    print('-' * 100)
print('The trainer applies English + int_score >= 3 + minimum text length filters.')

## 2. Start real-data pretraining

The trainer creates a tokenizer from training documents only, packs contiguous token streams with EOS document boundaries, computes next-token cross-entropy, adds the ADA MoE load-balancing auxiliary loss, evaluates on held-out documents, and writes resumable checkpoints.

**Resume behavior:** first run uses `RESUME = False`. If Colab disconnects after a checkpoint is saved, set `RESUME = True` and rerun this cell. To extend a completed run, increase `MAX_STEPS` and set `RESUME = True`.

In [ ]:
MAX_STEPS = 500
BATCH_SIZE = 4
SEQUENCE_LENGTH = 256
VOCAB_SIZE = 8192
TOKENIZER_DOCS = 20000
RESUME = False  # Set True only to continue an existing checkpoint in RUN_DIR.

command = [
    sys.executable, 'scripts/ada_pretrain_fineweb_edu.py',
    '--output-dir', str(RUN_DIR),
    '--max-steps', str(MAX_STEPS),
    '--batch-size', str(BATCH_SIZE),
    '--sequence-length', str(SEQUENCE_LENGTH),
    '--vocab-size', str(VOCAB_SIZE),
    '--tokenizer-docs', str(TOKENIZER_DOCS),
    '--eval-every', '100',
    '--eval-batches', '8',
    '--save-every', '100',
    '--device', 'cuda',
]
if RESUME:
    command.append('--resume')
print('Launching:', ' '.join(command))
subprocess.run(command, check=True)

## 3. Inspect pretraining metrics and checkpoint

The plotted curve is from actual FineWeb-Edu training batches and held-out validation batches. If validation loss rises while training loss falls, stop and investigate overfitting, data split, learning rate, or training duration.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

metrics_path = RUN_DIR / 'metrics.jsonl'
metrics = pd.read_json(metrics_path, lines=True)
display(metrics.tail(12))
plot_data = metrics.dropna(subset=['validation_loss'])
plt.figure(figsize=(10, 5))
if 'train_loss' in plot_data:
    plt.plot(plot_data['step'], plot_data['train_loss'], marker='o', label='Train loss')
plt.plot(plot_data['step'], plot_data['validation_loss'], marker='o', label='Validation loss')
plt.xlabel('Optimizer step')
plt.ylabel('Next-token cross-entropy')
plt.title('ADA real-data pretraining')
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

manifest = json.loads((RUN_DIR / 'run_manifest.json').read_text())
print(json.dumps(manifest, indent=2))
print('Checkpoint:', RUN_DIR / 'model' / 'model.pt')

## 4. Sample the pretrained checkpoint

This is a qualitative sanity check only. A 512k-token pilot is far too small to expect polished generations. The checkpoint and validation curve—not a cherry-picked sample—are the primary artifacts.

In [ ]:
from tokenizers import Tokenizer
from ada.neural import ADAForCausalLM

device = torch.device('cuda')
model = ADAForCausalLM.from_pretrained(RUN_DIR / 'model', map_location='cpu').to(device)
model.eval()
tokenizer = Tokenizer.from_file(str(RUN_DIR / 'tokenizer.json'))
eos_id = tokenizer.token_to_id('<|eos|>')

@torch.no_grad()
def generate(prompt, new_tokens=100, temperature=0.8, top_k=40):
    ids = tokenizer.encode(prompt).ids
    if not ids:
        raise ValueError('Prompt produced no tokens')
    for _ in range(new_tokens):
        context = ids[-model.config.max_seq_len:]
        x = torch.tensor([context], dtype=torch.long, device=device)
        logits = model(x).logits[0, -1].float() / temperature
        values, indices = torch.topk(logits, k=min(top_k, logits.numel()))
        probs = torch.softmax(values, dim=-1)
        next_id = int(indices[torch.multinomial(probs, 1)].item())
        if next_id == eos_id:
            break
        ids.append(next_id)
    return tokenizer.decode(ids, skip_special_tokens=True)

prompt = 'A scientific explanation of how energy is transferred in a mechanical system:'
print(generate(prompt))

## 5. Preserve the run

The Drive folder contains `tokenizer.json`, `model/config.json`, `model/model.pt`, `training_state.pt`, `metrics.jsonl`, and `run_manifest.json`. Keep the manifest with every checkpoint so the dataset, license declaration, tokenizer, architecture, split rule, and token budget remain traceable.

**Next scale-up gate:** inspect validation behavior, throughput, peak VRAM, checkpoint integrity, and corpus quality before increasing context length, model width, token budget, or adding code/multilingual/domain corpora.